# MuSeg-AI Thigh Segmentation — Water (Lambda)

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (`thigh-model3`)
on **Dixon WATER** stacks.

The model expects two Dixon channels (in-phase + out-of-phase). The WATER volume is passed
as both channels — the same approximation used for fat-fraction.

The setup cell installs Docker (if absent) and museg-ai. The first segmentation run will
pull the `fabianbalsiger/museg:thigh-model3` Docker image (~several GB).

## Before running — upload to Lambda

```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  your machine9:~/
```

## Download results when done

```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine9:~/museg_thigh_water_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/museg_thigh_water_segs/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        print(f'[WARN] {cmd[:80]}: {out[:300]}')
    else:
        print(f'OK: {cmd[:60]}')
    return r.returncode == 0

# ── Docker ───────────────────────────────────────────────────────────────────
r = subprocess.run('which docker', shell=True, capture_output=True)
if r.returncode != 0:
    print('Installing docker.io ...')
    sh('sudo apt-get update -qq')
    sh('sudo apt-get install -y docker.io')
else:
    print('Docker already present:', r.stdout.strip())

sh('sudo systemctl start docker')
# Open socket so current user can call Docker without sudo
sh('sudo chmod 666 /var/run/docker.sock')

# ── museg-ai ─────────────────────────────────────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/fabianbalsiger/museg-ai.git'])

import importlib; importlib.invalidate_caches()
import musegai
print('museg-ai version:', musegai.__version__)

In [ ]:
import docker
try:
    client = docker.from_env()
    client.ping()
    print('Docker is running.')
except Exception as e:
    raise RuntimeError(
        'Docker is not reachable. Re-run the setup cell, then retry.\n'
        f'Original error: {e}'
    )

In [ ]:
import glob
import os
import numpy as np
from musegai import api

DATA_ROOT  = os.path.expanduser('~/myosegmenTUM')
IMAGE_GLOB = os.path.join(DATA_ROOT, '*', 'ImageData', '*_WATER', '*_WATER_stack*.nii')
OUTPUT_DIR = os.path.expanduser('~/museg_thigh_water_segs')

LABEL_MAP = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Data root  : {os.path.abspath(DATA_ROOT)}')
print(f'Output dir : {os.path.abspath(OUTPUT_DIR)}')
print(f'Found      : {len(image_files)} WATER stacks')
for p in image_files:
    print(' ', p)

In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_museg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    vol = api.Volume.load(nii_path)
    print(f'  Shape: {vol.shape}  Spacing: {vol.spacing}')

    # museg-ai expects {name: [ch0, ch1]} (in-phase + out-of-phase).
    # Water image is passed as both channels — known approximation.
    results, labels = api.segment_volumes(
        {stem: [vol, vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[stem]
    segmentation.save(out_path)
    print(f'  Saved -> {out_path}')

    seg_arr = segmentation.array
    print(f'  Labels present: {sorted(np.unique(seg_arr).tolist())}')
    print(f"  {'Label':<6} {'Muscle':<25} {'Voxels':>10}")
    print(f"  {'-'*45}")
    for idx, name in LABEL_MAP.items():
        n = int((seg_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nAll done.')

In [ ]:
# Sanity check — reload one result
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.nii.gz')))
print(f'Total output files: {len(results)}')
if results:
    sample = api.Volume.load(results[0])
    print('Sample:', results[0])
    print('  Shape  :', sample.shape)
    print('  Spacing:', sample.spacing)
    labels_present = sorted(np.unique(sample.array).tolist())
    print('  Labels :', labels_present)
    for idx in labels_present:
        if idx > 0:
            print(f'    {idx} {LABEL_MAP.get(idx, "unknown"):<25} {int((sample.array == idx).sum()):>10,} voxels')
else:
    print('No results yet.')